In [2]:
from pathlib import Path
import sys

import numpy as np
import torch

# Locate awdro from the notebook's working directory.
current = Path.cwd().resolve()

root = next(
    (
        folder
        for folder in (current, *current.parents)
        if (folder / "inner maximization" / "RectangularSolver.py").is_file()
    ),
    None,
)

if root is None:
    raise FileNotFoundError(
        "Open this notebook inside your awdro project folder."
    )

sys.path.insert(0, str(root / "inner maximization"))

from RectangularSolver import RectangularSolver
from NonrectangularSolver import NonrectangularSolver

In [3]:
import sys
import numpy as np
import torch

sys.path.insert(0, str(root / "inner maximization"))

from RectangularSolver import RectangularSolver
from NonrectangularSolver import NonrectangularSolver


class HullToyModel:
    """Two equiprobable outcomes at each step, with independent periods."""

    s0 = 2.0
    v0 = 0.0

    def is_valid_state(self, state):
        state = np.asarray(state, dtype=float)

        return (
            state.shape == (2,)
            and np.isfinite(state).all()
            and state[0] > 0
            and state[1] >= 0
        )

    def step(self, state, dt, rng):
        if state.shape != (2, 2):
            raise ValueError("Use exactly two reference branches.")

        # First-period stock: 1 or 3.
        # Second-period stock: independently 1 or 9.
        stocks = [1.0, 3.0] if state[0, 0] == 2.0 else [1.0, 9.0]

        return np.column_stack([stocks, np.zeros(2)])


def make_test_solver(solver_class, budget, n_steps=2):
    return solver_class(
        reference_model=HullToyModel(),
        trading_times=np.linspace(0.0, 1.0, n_steps + 1),
        budget=budget,
        discretization={
            "reference_branches": 2,
            "seed": 42,
        },
    )


def constant_loss(paths):
    return torch.full(
        (paths.shape[0],),
        7.0,
        dtype=paths.dtype,
        device=paths.device,
    )


def terminal_stock_loss(paths):
    return paths[:, -1, 0]


def matching_loss(paths):
    first_high = paths[:, 1, 0] > 2.0
    second_high = paths[:, 2, 0] > 5.0

    return 10.0 * (first_high == second_high).to(paths.dtype)

In [4]:
for budget in [0.0, 0.5, 1.0]:
    solver = make_test_solver(RectangularSolver, budget)

    np.testing.assert_allclose(
        solver.solve(constant_loss)["value"],
        7.0,
        rtol=0.0,
        atol=1e-8,
    )

zero_solver = make_test_solver(RectangularSolver, budget=0.0)

np.testing.assert_allclose(
    zero_solver.solve(matching_loss)["value"],
    5.0,
    rtol=0.0,
    atol=1e-8,
)

# Also exercise the actual hull recursion at zero budget.
# This bypasses solve's nominal-expectation shortcut.
initial_history = np.array([[2.0, 0.0]])

np.testing.assert_allclose(
    zero_solver.backward_value(initial_history, matching_loss),
    5.0,
    rtol=0.0,
    atol=1e-8,
)

print("Constant-loss and zero-budget checks passed.")

Constant-loss and zero-budget checks passed.


In [5]:
for budget in [0.0, 0.5, 1.0, 2.0]:
    rectangular = make_test_solver(
        RectangularSolver, budget, n_steps=1
    )
    nonrectangular = make_test_solver(
        NonrectangularSolver, budget, n_steps=1
    )

    rectangular_value = rectangular.solve(terminal_stock_loss)["value"]
    nonrectangular_value = nonrectangular.solve(terminal_stock_loss)["value"]

    expected = min(2.0 + budget**2 / 2.0, 3.0)

    np.testing.assert_allclose(
        rectangular_value, expected, rtol=0.0, atol=1e-8
    )
    np.testing.assert_allclose(
        nonrectangular_value, expected, rtol=0.0, atol=2e-5
    )

    print(
        f"budget={budget:.2f}  "
        f"rectangular={rectangular_value:.8f}  "
        f"nonrectangular={nonrectangular_value:.8f}"
    )

print("One-period equality check passed.")

budget=0.00  rectangular=2.00000000  nonrectangular=2.00000000
budget=0.50  rectangular=2.12500000  nonrectangular=2.12500000
budget=1.00  rectangular=2.50000000  nonrectangular=2.50000000
budget=2.00  rectangular=3.00000000  nonrectangular=3.00000000
One-period equality check passed.


In [6]:
for budget in [0.0, 0.5, 1.0]:
    rectangular = make_test_solver(RectangularSolver, budget)
    nonrectangular = make_test_solver(NonrectangularSolver, budget)

    # Confirm that both solvers use identical discretizations.
    np.testing.assert_array_equal(
        np.asarray([
            node["state"] for node in rectangular.reference_nodes
        ]),
        np.asarray([
            node["state"] for node in nonrectangular.reference_nodes
        ]),
    )

    for t in range(3):
        np.testing.assert_array_equal(
            rectangular.adversarial_candidates[t],
            nonrectangular.adversarial_candidates[t],
        )

    rectangular_value = rectangular.solve(matching_loss)["value"]
    nonrectangular_value = nonrectangular.solve(matching_loss)["value"]

    expected_rectangular = 5.0 + 10.0 * budget**2 / 32.0
    expected_nonrectangular = 5.0 + 10.0 * budget**2 / 64.0
    expected_gap = 10.0 * budget**2 / 64.0

    np.testing.assert_allclose(
        rectangular_value,
        expected_rectangular,
        rtol=0.0,
        atol=1e-8,
    )
    np.testing.assert_allclose(
        nonrectangular_value,
        expected_nonrectangular,
        rtol=0.0,
        atol=2e-5,
    )
    np.testing.assert_allclose(
        rectangular_value - nonrectangular_value,
        expected_gap,
        rtol=0.0,
        atol=2e-5,
    )

    print(
        f"budget={budget:.2f}  "
        f"rectangular={rectangular_value:.8f}  "
        f"nonrectangular={nonrectangular_value:.8f}  "
        f"gap={rectangular_value - nonrectangular_value:.8f}"
    )

print("Known rectangularity-gap check passed.")

budget=0.00  rectangular=5.00000000  nonrectangular=5.00000000  gap=0.00000000
budget=0.50  rectangular=5.07812500  nonrectangular=5.03906257  gap=0.03906243
budget=1.00  rectangular=5.31250000  nonrectangular=5.15625003  gap=0.15624997
Known rectangularity-gap check passed.


In [7]:
sys.path.insert(0, str(root / "path simulation"))

from Heston import Heston

model = Heston(
    s0=100.0,
    v0=0.04,
    mu=0.0,
    kappa=2.0,
    theta=0.04,
    vol_of_vol=0.3,
    rho=-0.7,
)


def hedging_loss(paths):
    stocks = paths[:, :, 0]

    # Holdings use only the current stock observation.
    holdings = (stocks[:, :-1] < 100.0).to(paths.dtype)
    gains = (
        holdings * (stocks[:, 1:] - stocks[:, :-1])
    ).sum(dim=1)

    payoff = torch.clamp(stocks[:, -1] - 100.0, min=0.0)

    return torch.clamp(payoff - 5.0 - gains, min=0.0)


def make_heston_solver(solver_class, budget, seed=42):
    return solver_class(
        reference_model=model,
        trading_times=[0.0, 0.5, 1.0],
        budget=budget,
        discretization={
            "reference_branches": 2,
            "seed": seed,
        },
    )

In [8]:
budgets = [0.0, 0.5, 1.0, 2.0]
tolerance = 1e-4

rectangular_values = []
nonrectangular_values = []

baseline = make_heston_solver(RectangularSolver, budget=0.0)

# Independent nominal expectation from complete reference paths.
reference_losses = hedging_loss(
    torch.tensor(baseline.reference_paths, dtype=torch.float64)
).numpy()

nominal = float(
    baseline.reference_probabilities @ reference_losses
)

# Every admissible law is supported on these adversarial paths.
candidate_losses = hedging_loss(
    torch.tensor(baseline.adversarial_paths, dtype=torch.float64)
).numpy()

maximum_loss = float(candidate_losses.max())

for budget in budgets:
    rectangular = make_heston_solver(RectangularSolver, budget)
    nonrectangular = make_heston_solver(NonrectangularSolver, budget)

    # Ensure the comparison uses identical reference trees.
    rectangular_states = np.asarray([
        node["state"] for node in rectangular.reference_nodes
    ])
    nonrectangular_states = np.asarray([
        node["state"] for node in nonrectangular.reference_nodes
    ])

    np.testing.assert_array_equal(
        rectangular_states, nonrectangular_states
    )
    np.testing.assert_array_equal(
        rectangular.reference_paths, baseline.reference_paths
    )

    for t in range(3):
        np.testing.assert_array_equal(
            rectangular.adversarial_candidates[t],
            nonrectangular.adversarial_candidates[t],
        )

    rect_value = float(rectangular.solve(hedging_loss)["value"])
    nonrect_value = float(nonrectangular.solve(hedging_loss)["value"])

    assert np.isfinite(rect_value)
    assert np.isfinite(nonrect_value)

    # Nominal law belongs to the original set, which belongs to its hull.
    assert nonrect_value >= nominal - tolerance
    assert rect_value >= nonrect_value - tolerance

    # No expected loss can exceed the largest supported loss.
    assert rect_value <= maximum_loss + tolerance
    assert nonrect_value <= maximum_loss + tolerance

    rectangular_values.append(rect_value)
    nonrectangular_values.append(nonrect_value)

    print(
        f"budget={budget:.2f}  "
        f"rectangular={rect_value:.8f}  "
        f"nonrectangular={nonrect_value:.8f}  "
        f"gap={rect_value - nonrect_value:.8f}"
    )

np.testing.assert_allclose(
    [rectangular_values[0], nonrectangular_values[0]],
    nominal,
    rtol=0.0,
    atol=tolerance,
)

# Increasing the budget enlarges each ambiguity set.
assert np.all(np.diff(rectangular_values) >= -tolerance)
assert np.all(np.diff(nonrectangular_values) >= -tolerance)

print("Heston comparison checks passed.")

budget=0.00  rectangular=6.47151148  nonrectangular=6.47151148  gap=0.00000000
budget=0.50  rectangular=6.71471202  nonrectangular=6.56298104  gap=0.15173098
budget=1.00  rectangular=7.19933186  nonrectangular=6.59289484  gap=0.60643702
budget=2.00  rectangular=8.83748025  nonrectangular=6.71255002  gap=2.12493023
Heston comparison checks passed.


In [9]:
shift = -10.0


def shifted_loss(paths):
    return hedging_loss(paths) + shift


for solver_class in [RectangularSolver, NonrectangularSolver]:
    solver = make_heston_solver(solver_class, budget=1.0)

    original = solver.solve(hedging_loss)["value"]
    shifted = solver.solve(shifted_loss)["value"]

    np.testing.assert_allclose(
        shifted,
        original + shift,
        rtol=0.0,
        atol=1e-4,
    )

    print(f"{solver_class.__name__}: translation check passed.")

RectangularSolver: translation check passed.
NonrectangularSolver: translation check passed.


In [10]:
scale = 3.0


def scaled_loss(paths):
    return scale * hedging_loss(paths)


for solver_class in [RectangularSolver, NonrectangularSolver]:
    solver = make_heston_solver(solver_class, budget=1.0)

    original = float(solver.solve(hedging_loss)["value"])
    scaled = float(solver.solve(scaled_loss)["value"])

    np.testing.assert_allclose(
        scaled,
        scale * original,
        rtol=0.0,
        atol=1e-4,
    )

    print(f"{solver_class.__name__}: scaling check passed.")

RectangularSolver: scaling check passed.
NonrectangularSolver: scaling check passed.


In [11]:
class BranchingTestModel:
    s0 = 2.0
    v0 = 0.0

    def is_valid_state(self, state):
        state = np.asarray(state, dtype=float)

        return (
            state.shape == (2,)
            and np.isfinite(state).all()
            and state[0] > 0
            and state[1] >= 0
        )

    def step(self, state, dt, rng):
        if state.shape != (2, 2):
            raise ValueError("Use exactly two branches.")

        stock = state[0, 0]

        if stock == 2.0:
            next_stocks = [1.0, 3.0]
        else:
            next_stocks = [2.0 * stock + 10.0, 2.0 * stock + 11.0]

        return np.column_stack([next_stocks, np.zeros(2)])


branch_solver = RectangularSolver(
    reference_model=BranchingTestModel(),
    trading_times=[0.0, 1.0, 2.0, 3.0],
    budget=0.0,
    discretization={
        "reference_branches": 2,
        "seed": 42,
    },
)

# From stock 1, the next stock is 12 or 13.
# Stock 16 exists at that date, but only after stock 3.
unreachable_history = np.array([
    [2.0, 0.0],
    [1.0, 0.0],
    [16.0, 0.0],
])

reachable_history = np.array([
    [2.0, 0.0],
    [1.0, 0.0],
    [12.0, 0.0],
])

constant_values = np.full(
    len(branch_solver.adversarial_candidates[3]),
    7.0,
)

assert branch_solver.conditional_worst_case(
    unreachable_history,
    constant_values,
) is None

reachable_value = branch_solver.conditional_worst_case(
    reachable_history,
    constant_values,
)

assert reachable_value is not None
np.testing.assert_allclose(
    reachable_value, 7.0, rtol=0.0, atol=1e-8
)

print("Reachable/unreachable history checks passed.")

Reachable/unreachable history checks passed.


In [12]:
rng = np.random.default_rng(123)

next_states = branch_solver.adversarial_candidates[3]
continuation_values = rng.normal(size=len(next_states))

successor_mask = np.isin(next_states[:, 0], [34.0, 35.0])
assert successor_mask.sum() == 2

expected = float(continuation_values[successor_mask].mean())

actual = branch_solver.conditional_worst_case(
    reachable_history,
    continuation_values,
)

np.testing.assert_allclose(
    actual, expected, rtol=0.0, atol=1e-8
)

print("Conditional-probability check passed.")

Conditional-probability check passed.


In [13]:
def signed_loss(paths):
    return paths[:, -1, 0] - 100.0


reference_losses = signed_loss(
    torch.tensor(
        branch_solver.reference_paths,
        dtype=torch.float64,
    )
).numpy()

expected = float(
    branch_solver.reference_probabilities @ reference_losses
)

actual = branch_solver.backward_value(
    np.array([[2.0, 0.0]]),
    signed_loss,
)

assert expected < 0.0

np.testing.assert_allclose(
    actual, expected, rtol=0.0, atol=1e-8
)

print(f"Expected: {expected:.8f}")
print(f"Computed: {actual:.8f}")
print("Negative-loss recursion check passed.")

Expected: -60.50000000
Computed: -60.50000000
Negative-loss recursion check passed.


In [14]:
solver = RectangularSolver(
    reference_model=BranchingTestModel(),
    trading_times=[0.0, 1.0, 2.0, 3.0],
    budget=0.0,
    discretization={"reference_branches": 2, "seed": 42},
)

x = solver.reference_paths
y = solver.adversarial_paths
p = solver.reference_probabilities

n = len(x)
m = len(y)

identity = np.zeros((n, m))

for i, reference_path in enumerate(x):
    matching_indices = np.flatnonzero(
        np.all(y == reference_path, axis=(1, 2))
    )

    assert matching_indices.size == 1
    identity[i, matching_indices[0]] = p[i]

flat_identity = identity.ravel()

np.testing.assert_allclose(
    solver.A_eq @ flat_identity,
    solver.b_eq,
    rtol=0.0,
    atol=1e-12,
)

identity_cost = float(
    np.sum(solver.transport_costs * identity)
)

np.testing.assert_allclose(
    identity_cost, 0.0, rtol=0.0, atol=1e-12
)

assert np.all(flat_identity >= 0.0)

print("Identity coupling is feasible and has zero cost.")

Identity coupling is feasible and has zero cost.


In [15]:
rare_history = np.array([
    [2.0, 0.0],
    [1.0, 0.0],
    [16.0, 0.0],
])

for budget in [0.1, 0.01, 0.001]:
    solver = RectangularSolver(
        reference_model=BranchingTestModel(),
        trading_times=[0.0, 1.0, 2.0, 3.0],
        budget=budget,
        discretization={"reference_branches": 2, "seed": 42},
    )

    # Confirm that the identity matrix from the previous cell still applies.
    np.testing.assert_array_equal(solver.reference_paths, x)
    np.testing.assert_array_equal(solver.adversarial_paths, y)
    np.testing.assert_array_equal(solver.reference_probabilities, p)

    history_mask = np.all(
        y[:, :3, :] == rare_history,
        axis=(1, 2),
    )

    # Select one complete adversarial path through the rare history.
    target_id = np.flatnonzero(history_mask)[0]

    # Cost of sending every reference path to that fixed target path.
    fixed_target_cost = float(
        p @ solver.transport_costs[:, target_id]
    )
    assert fixed_target_cost > 0.0

    # Choose a positive mixture weight using at most half the budget.
    alpha = min(
        0.5,
        budget**2 / (2.0 * fixed_target_cost),
    )

    witness = (1.0 - alpha) * identity
    witness[:, target_id] += alpha * p

    # Verify this explicit coupling against the original constraints.
    np.testing.assert_allclose(
        solver.A_eq @ witness.ravel(),
        solver.b_eq,
        rtol=0.0,
        atol=1e-12,
    )

    witness_cost = float(
        np.sum(solver.transport_costs * witness)
    )
    assert witness_cost <= budget**2 + 1e-12

    history_probability = float(witness[:, history_mask].sum())
    assert history_probability > 0.0

    # A constant continuation value must remain constant,
    # even when the history is extremely rare.
    continuation_values = np.full(
        len(solver.adversarial_candidates[3]),
        7.0,
    )

    value = solver.conditional_worst_case(
        rare_history,
        continuation_values,
    )

    assert value is not None, (
        "The optimizer reported an unreachable history, "
        "but we constructed a feasible coupling reaching it."
    )

    np.testing.assert_allclose(
        value, 7.0, rtol=0.0, atol=1e-6
    )

    print(
        f"budget={budget:.3g}  "
        f"witness_probability={history_probability:.3e}  "
        f"conditional_value={value:.8f}"
    )

print("Rare-history checks passed.")

budget=0.1  witness_probability=8.929e-05  conditional_value=7.00000000
budget=0.01  witness_probability=8.929e-07  conditional_value=7.00000000
budget=0.001  witness_probability=8.929e-09  conditional_value=7.00000000
Rare-history checks passed.
